In [6]:
import cv2
import numpy as np
import math

roi = None
drawing = False
x0,y0 = -1,-1


# -----------------------------
# Mouse patch selector
# -----------------------------
def select_patch(event,x,y,flags,param):

    global roi,drawing,x0,y0,frame_copy

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        x0,y0 = x,y

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        img = frame_copy.copy()
        cv2.rectangle(img,(x0,y0),(x,y),(0,255,0),2)
        cv2.imshow("Select Green Patch",img)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        roi = (x0,y0,x,y)


# -----------------------------
# RGB → CbCr
# -----------------------------
def rgb_to_cbcr(frame):

    r = frame[:,:,2].astype(np.float32)
    g = frame[:,:,1].astype(np.float32)
    b = frame[:,:,0].astype(np.float32)

    cb = 128 + (-0.168736*r -0.331264*g +0.5*b)
    cr = 128 + (0.5*r -0.418688*g -0.081312*b)

    return cb,cr


# -----------------------------
# Compute chroma mask
# -----------------------------
def compute_mask(frame, cb_key, cr_key, tola, tolb):

    cb,cr = rgb_to_cbcr(frame)

    dist = np.sqrt((cb_key-cb)**2 + (cr_key-cr)**2)

    mask = np.zeros_like(dist)

    near = dist < tola
    mid  = (dist >= tola) & (dist < tolb)
    far  = dist >= tolb

    mask[near] = 0
    mask[mid]  = (dist[mid]-tola)/(tolb-tola)
    mask[far]  = 1

    mask = 1-mask

    return mask


# -----------------------------
# Remove green spill
# -----------------------------
def remove_color_cast(frame, mask, key_color):

    r_key,g_key,b_key = key_color

    frame = frame.astype(np.float32)

    frame[:,:,2] = np.maximum(frame[:,:,2] - mask*r_key,0)
    frame[:,:,1] = np.maximum(frame[:,:,1] - mask*g_key,0)
    frame[:,:,0] = np.maximum(frame[:,:,0] - mask*b_key,0)

    return frame


# -----------------------------
# Composite
# -----------------------------
def composite(fg,bg,mask):

    mask = mask[:,:,None]

    out = fg*(1-mask) + bg*mask

    return np.clip(out,0,255).astype(np.uint8)


# -----------------------------
# Main
# -----------------------------
cap = cv2.VideoCapture("data/greenscreen-asteroid.mp4")
bg_cap = cv2.VideoCapture("data/background.mp4")

ret,frame = cap.read()

frame_copy = frame.copy()

cv2.namedWindow("Select Green Patch")
cv2.setMouseCallback("Select Green Patch",select_patch)

while roi is None:
    cv2.imshow("Select Green Patch",frame_copy)
    if cv2.waitKey(1) == 27:
        break

x0,y0,x1,y1 = roi
patch = frame[y0:y1,x0:x1]

# key color
mean_color = patch.reshape(-1,3).mean(axis=0)
b_key,g_key,r_key = mean_color

cb_key = 128 + (-0.168736*r_key -0.331264*g_key +0.5*b_key)
cr_key = 128 + (0.5*r_key -0.418688*g_key -0.081312*b_key)


# UI controls
cv2.namedWindow("Controls")

cv2.createTrackbar("tola","Controls",10,100,lambda x:None)
cv2.createTrackbar("tolb","Controls",40,200,lambda x:None)
cv2.createTrackbar("colorcast","Controls",50,100,lambda x:None)


# process video
while True:

    ret,frame = cap.read()
    ret2,bg = bg_cap.read()

    if not ret or not ret2:
        break

    bg = cv2.resize(bg,(frame.shape[1],frame.shape[0]))

    tola = cv2.getTrackbarPos("tola","Controls")
    tolb = cv2.getTrackbarPos("tolb","Controls")
    cast = cv2.getTrackbarPos("colorcast","Controls")/100.0

    mask = compute_mask(frame,cb_key,cr_key,tola,tolb)

    fg = remove_color_cast(frame,mask*cast,(r_key,g_key,b_key))

    result = composite(fg,bg,mask)

    cv2.imshow("ChromaKey Output",result)

    if cv2.waitKey(1) & 0xFF == 27:
        break


cap.release()
bg_cap.release()
cv2.destroyAllWindows()